# Day4 Lab4：给职业数字人加上 Context

欢迎来到 Day 4 的 Lab。

前三天，我们已经让职业数字人完成了三次升级：

- Day 1：接上 LLM，让它有了“大脑”
- Day 2：加上 workflow，让它知道不同任务走不同路
- Day 3：接上 tool，让它可以读取、分析、搜索和推送

今天，我们要给它加上第四块能力：

**context。**

也就是让它在回答之前，先看到这次任务真正需要的信息。

## 1. 先复习一下：什么是 Context

今天我们讲过一句很重要的话：

**context = 模型这一次看到的全部信息。**

这里的“这一次看到”，非常关键。

它不是模型永远知道的所有东西，  
也不是你电脑里的所有文件，  
而是这一次调用模型时，真正被放进 prompt 里的信息。

在今天的最小系统里，context 主要来自三部分：

```text
history  = 当前对话前面聊过什么
memory   = 长期需要记住的身份、偏好、规则和边界
resource = 外部资料里有什么
```
今天我们要做的，就是把这三类信息组织起来，交给模型使用。

---

## 2. RAG 只负责处理 Resource

这里要先分清楚一件事：

**RAG 不是 memory。**  
**RAG 也不是 history。**

RAG 只负责一件事：

**从 resource 里找到和当前问题最相关的资料。**

也就是说：

```text
history  可以直接进入 context
memory   可以直接进入 context
resource 通常要先经过 RAG 检索，再进入 context
```

所以今天的完整结构是：
当前问题
+ history
+ memory
+ retrieved resource
= final context

最后，模型会基于这个 final context 来回答。

---



## 3. 今天先使用 Context，不做自动更新

在真实系统里，history、memory、resource 都可能会被更新。

但它们的更新方式不一样。

- **history**：通常会随着每轮对话自动追加。
- **memory**：不能随便自动写入，因为它代表长期身份、偏好、权限和边界，最好经过规则判断或用户确认。
- **resource**：一般不是每轮对话都更新，通常是在上传新资料、修改模板、更新制度时才维护。

所以在 Lab4 里，我们先不做自动更新。

今天先把三类信息提前准备好，重点跑通一件事：

**AI 如何读取 history、memory、resource，并把它们组装进 context。**

自动维护这些信息，是后面更完整系统要做的事。

---

## 4. 今天要做什么

今天我们要做的是：

```text
职业数字人 v3
= 能接住 history
+ 能参考 memory
+ 能从 resource 里找资料
+ 能把三者组装进 context 再回答
```

今天的 Lab 会分成四部分：

复习概念，准备环境
读取 history 和 memory
给 resource 加上最小 RAG
组装 final context，并让模型回答

这一节先完成第一部分：准备工作。

---

## 5. 今天用到的工具包

这一节我们会用到几个很基础的工具包：

- `os`：读取环境变量
- `json`：读取 memory 文件
- `Path`：更方便地处理文件路径
- `numpy`：后面计算相似度
- `dotenv`：读取 `.env` 里的 API key
- `OpenAI`：用 OpenAI 兼容格式调用大模型
- `display` 和 `Markdown`：在 Notebook 里更清楚地展示结果

先不用记住每一个工具包的细节。

你只要知道：

**这一部分是在把今天的厨房准备好。**

In [3]:
# 这一段是在导入今天会用到的工具包

import os
import json
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

## 6. 读取 `.env` 配置，链接大模型

在运行下面的代码前，请确认你的 `.env` 文件里已经有这些内容：

```text
API_KEY=你的模型 API key
BASE_URL=你的模型服务地址
MODEL_NAME=你的对话模型名
EMBEDDING_MODEL=你的 embedding 模型名
```

MODEL_NAME 用来生成回答。

EMBEDDING_MODEL 用来把文字变成数字表示。
后面做 RAG 的时候，我们会用它来比较“问题”和“资料”是否相关。

In [4]:
# 这一段是在读取 .env，并连接大模型

load_dotenv()

API_KEY = os.getenv("API_KEY")
BASE_URL = os.getenv("BASE_URL")
MODEL_NAME = os.getenv("MODEL_NAME")
EMBEDDING_MODEL_NAME = os.getenv("EMBEDDING_MODEL_NAME")

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

## 7. 封装 `call_llm()`

接下来，我们封装一个通用函数：

```python
call_llm()
```

这个函数只有一个作用：

把问题交给模型，并拿回回答。

后面我们会把 history、memory、retrieved resource 组装成 context，放进 system_prompt 里。

这样模型回答时，就不是凭空回答，而是带着上下文回答。

In [7]:
# 这一段是在封装一个通用的 LLM 调用函数

def call_llm(user_question: str, system_prompt: str = "") -> str:
    """
    调用大模型，返回文本回答。

    参数：
    - user_question：用户这一次提出的问题
    - system_prompt：系统提示词，也可以放入整理好的 context

    返回：
    - 模型生成的文本
    """

    messages = []

    if system_prompt:
        messages.append({
            "role": "system",
            "content": system_prompt
        })

    messages.append({
        "role": "user",
        "content": user_question
    })

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages
    )

    return response.choices[0].message.content

## 8. 测试 LLM 是否可以正常回答

函数写好以后，我们先做一个最小测试。

这一步是为了确认：

**模型已经连通，可以正常返回结果。**

In [9]:
# 这一段是在测试 LLM 是否可以正常回答

test_answer = call_llm(
    user_question="请用一句话解释什么是 context。",
    system_prompt="你是一位清晰、适合零基础学员的 AI 课程老师。"
)

display(Markdown(f"""
### 🤖 LLM 测试结果

{test_answer}
"""))


### 🤖 LLM 测试结果

Context（上下文）就像对话的“前情提要”，是你提供给 AI 的背景信息，帮助它准确理解你的意图并给出连贯、合适的回答。


## 9. 封装 `get_embedding()`

接下来，我们再封装一个新函数：

```python
get_embedding()
```

这个函数是 Day4 新增的。

它的作用是：

把一段文字转换成一组数字。

为什么要这样做？

因为后面做 RAG 时，我们需要判断：

用户的问题，和哪几段资料更相关。

电脑不能直接像人一样理解“意思接近”。

所以我们先把文字变成数字表示，
再用数字去计算相似度。

你可以先这样理解：

embedding = 文字的数字坐标。

In [10]:
# 这一段是在封装 embedding 函数
# embedding 会在后面的 RAG 检索里使用

def get_embedding(text: str) -> list:
    """
    把一段文字转换成 embedding。

    参数：
    - text：需要转换的文本

    返回：
    - 一组数字向量
    """

    response = client.embeddings.create(
        model=EMBEDDING_MODEL_NAME,
        input=text
    )

    return response.data[0].embedding

## 10. 测试 embedding 是否可以正常生成

现在我们用一句话测试一下 embedding。

如果成功，系统会返回一组数字。

这组数字本身不需要我们看懂。

我们只需要确认：

**文字已经可以被转换成向量。**

In [11]:
# 这一段是在测试 embedding 是否能正常生成

test_embedding = get_embedding("请把刚才的数据分析结果整理成项目群消息。")

display(Markdown(f"""
### ✅ Embedding 测试成功

这段文字已经被转换成一组数字。

向量长度：

```text
{len(test_embedding)}
前 5 个数字示例：

{test_embedding[:5]}

"""))


### ✅ Embedding 测试成功

这段文字已经被转换成一组数字。

向量长度：

```text
1024
前 5 个数字示例：

[-0.0688682571053505, -0.016837624832987785, -0.029011305421590805, -0.0617537647485733, 0.012213206849992275]



# Part 2：读取 History 和 Memory，搭出基础 Context

上一部分，我们已经完成了准备工作：

- 接好了大模型
- 封装了 `call_llm()`
- 封装了 `get_embedding()`
- 准备好了 context 文件路径

这一部分，我们先不做 RAG。

我们先处理两类 context：

```text
history = 当前对话前面发生过什么
memory  = 长期需要记住的角色、偏好、规则和边界
```

这一部分的目标是：

先让模型接住“刚才发生了什么”和“这个用户是谁”。

## 1. 读取 History

history 记录的是当前对话前面发生过什么。

比如：

用户上传了什么数据，  
工具分析出了什么结果，  
用户接下来想把结果发到哪里。

在真实系统里，history 通常会随着对话自动追加。

但在今天的 Lab 里，我们先用提前准备好的假数据文件。

In [14]:
# 统一定义路径，再读取
BASE_DIR = Path("data/context")

HISTORY_DIR = BASE_DIR / "history"
MEMORY_DIR = BASE_DIR / "memory"
RESOURCE_DIR = BASE_DIR / "resource"

history_file = HISTORY_DIR / "day3_analysis_history.txt"
memory_file = MEMORY_DIR / "user_profile_memory.json"

resource_files = [
    RESOURCE_DIR / "project_message_template.txt",
    RESOURCE_DIR / "metric_explanation.txt",
    RESOURCE_DIR / "communication_policy.txt"
]

In [15]:
# 这一段是在读取 history 文件

def load_text_file(file_path):
    """
    读取一个文本文件，并返回里面的文字内容。
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"找不到文件：{file_path}")

    return file_path.read_text(encoding="utf-8")


history_text = load_text_file(history_file)

display(Markdown(f"""
### ✅ History 读取成功

```text
{history_text}
"""))


### ✅ History 读取成功

```text
Day3 数据分析对话历史

用户上传了一份销售数据表，文件名为 sales_data.csv。

数据分析工具完成了基础分析，得到以下结果：
1. 本月整体销售额较上月上涨 12.5%。
2. 华东区销售额增长最快，环比增长 18.2%。
3. 华东区退货率也明显偏高，达到 7.8%，高于其他区域平均水平。
4. 华南区销售额稳定，但客单价略有下降。
5. 数据分析工具建议继续关注华东区退货原因，尤其是产品批次、渠道反馈和售后记录。

用户接着说：
“把刚才的数据分析结果，整理一下发到项目群里。”

注意：
这份 history 只是为了 Lab4 演示。
它代表当前对话前面已经发生过的内容。



## 2. 读取 Memory

memory 记录的是长期需要记住的信息。

它不只是语气偏好。

它也可以包括：

- 用户是什么角色
- 用户通常怎么表达
- 用户有哪些长期任务
- 哪些内容适合公开说
- 哪些内容不适合发到群里

今天我们用一个 JSON 文件保存 memory。

In [16]:
# 这一段是在读取 memory 文件

def load_json_file(file_path):
    """
    读取一个 JSON 文件，并返回 Python 字典。
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"找不到文件：{file_path}")

    with file_path.open("r", encoding="utf-8") as f:
        return json.load(f)


memory = load_json_file(memory_file)

display(Markdown(f"""
### ✅ Memory 读取成功

```json
{json.dumps(memory, ensure_ascii=False, indent=2)}
"""))


### ✅ Memory 读取成功

```json
{
  "user_profile": {
    "role": "项目负责人",
    "department": "数字化与业务协同团队",
    "work_context": "经常需要把数据分析结果整理成适合项目群、领导群或跨部门会议使用的简短结论。",
    "communication_style": "简洁、正式、清楚，不夸张，不写空话。"
  },
  "long_term_preferences": {
    "message_style": "先说结论，再说风险，最后说建议动作。",
    "preferred_length": "适合发群消息，控制在 150 字以内。",
    "tone": "稳重、专业、便于团队快速理解。"
  },
  "boundaries": {
    "sensitive_information": [
      "客户名称",
      "内部成本",
      "利润率",
      "具体异常明细",
      "未经确认的责任归因",
      "个人信息"
    ],
    "rules": [
      "敏感数据不要直接发到群里。",
      "涉及客户、成本、利润和异常明细时，只做概括表达。",
      "没有确认原因前，不要直接判断责任。",
      "如果结论来自初步分析，要保留谨慎表达。"
    ]
  }
}



## 3. 准备当前问题

现在我们准备一个当前问题。

这个问题接着 Day3 的场景：

前面已经做过数据分析，  
现在用户希望把分析结果整理成一段适合发到项目群的消息。

注意这里的“刚才”，就需要 history 来接住。

In [17]:
# 这一段是在准备用户当前的问题

current_question = "请把刚才的数据分析结果整理成一段适合发到项目群的消息。"

display(Markdown(f"""
### 当前问题

```text
{current_question}
```
"""))


### 当前问题

```text
请把刚才的数据分析结果整理成一段适合发到项目群的消息。
```


## 4. 拼出 Basic Context

现在我们先拼一个最基础的 context。

这一版还不包含 resource，  
也还没有 RAG。

它只包含：

```text
当前问题 + history + memory

这样模型至少能知道：

刚才发生了什么
用户是什么角色
回答时要注意哪些表达边界
```

In [18]:

# 这一段是在把 current question、history 和 memory 拼成 basic context

def format_memory(memory: dict) -> str:
    """
    把 memory 字典整理成更适合放进 prompt 的文字。
    """
    lines = []

    for key, value in memory.items():
        if isinstance(value, list):
            value_text = "；".join(str(item) for item in value)
        else:
            value_text = str(value)

        lines.append(f"- {key}: {value_text}")

    return "\n".join(lines)


def build_basic_context(current_question: str, history_text: str, memory: dict) -> str:
    """
    构建基础 context。
    
    这一版 context 只包含：
    - 当前问题
    - history
    - memory
    """
    memory_text = format_memory(memory)

    basic_context = f"""
    你是一个职业数字人助手。

    请你根据下面的 context 回答用户问题。

    【当前问题】
    {current_question}

    【History：当前对话前面发生过什么】
    {history_text}

    【Memory：长期需要记住的信息】
    {memory_text}

    请注意：
    1. 回答要基于上面的 context。
    2. 如果涉及群消息，请注意用户角色、表达风格和信息边界。
    3. 不要展开敏感细节，不要编造 context 里没有的信息。
    """

    return basic_context.strip()


basic_context = build_basic_context(
    current_question=current_question,
    history_text=history_text,
    memory=memory
)

display(Markdown(f"""
### ✅ Basic Context 已生成

```text
{basic_context}
"""))


### ✅ Basic Context 已生成

```text
你是一个职业数字人助手。

    请你根据下面的 context 回答用户问题。

    【当前问题】
    请把刚才的数据分析结果整理成一段适合发到项目群的消息。

    【History：当前对话前面发生过什么】
    Day3 数据分析对话历史

用户上传了一份销售数据表，文件名为 sales_data.csv。

数据分析工具完成了基础分析，得到以下结果：
1. 本月整体销售额较上月上涨 12.5%。
2. 华东区销售额增长最快，环比增长 18.2%。
3. 华东区退货率也明显偏高，达到 7.8%，高于其他区域平均水平。
4. 华南区销售额稳定，但客单价略有下降。
5. 数据分析工具建议继续关注华东区退货原因，尤其是产品批次、渠道反馈和售后记录。

用户接着说：
“把刚才的数据分析结果，整理一下发到项目群里。”

注意：
这份 history 只是为了 Lab4 演示。
它代表当前对话前面已经发生过的内容。


    【Memory：长期需要记住的信息】
    - user_profile: {'role': '项目负责人', 'department': '数字化与业务协同团队', 'work_context': '经常需要把数据分析结果整理成适合项目群、领导群或跨部门会议使用的简短结论。', 'communication_style': '简洁、正式、清楚，不夸张，不写空话。'}
- long_term_preferences: {'message_style': '先说结论，再说风险，最后说建议动作。', 'preferred_length': '适合发群消息，控制在 150 字以内。', 'tone': '稳重、专业、便于团队快速理解。'}
- boundaries: {'sensitive_information': ['客户名称', '内部成本', '利润率', '具体异常明细', '未经确认的责任归因', '个人信息'], 'rules': ['敏感数据不要直接发到群里。', '涉及客户、成本、利润和异常明细时，只做概括表达。', '没有确认原因前，不要直接判断责任。', '如果结论来自初步分析，要保留谨慎表达。']}

    请注意：
    1. 回答要基于上面的 context。
    2. 如果涉及群消息，请注意用户角色、表达风格和信息边界。
    3. 不要展开敏感细节，不要编造 context 里没有的信息。


## 5. 对比测试：没有 Context vs 有 Context

接下来，我们做一个小测试。

同一个问题，问两次。

第一次，不给 context。  
第二次，给 basic context。

你会看到：

没有 context 时，模型不知道“刚才”是什么。  
有 context 时，模型能接住前面的分析结果，也会注意用户的角色和边界。

In [19]:
# 这一段是在测试：不给 context，模型会怎么回答

answer_without_context = call_llm(
    user_question=current_question,
    system_prompt="你是一个简洁、清晰的 AI 助手。"
)

display(Markdown(f"""
### 🤖 没有 Context 的回答

{answer_without_context}
"""))


### 🤖 没有 Context 的回答

由于当前对话缺少具体的分析数据，我为你准备了一个**开箱即用的群消息模板**。你只需替换括号内容即可直接发送。如需我直接生成完整文案，请回复核心数据或结论。

---
【数据同步】📊 [项目/模块名称] - [分析主题]
各位好，本次[周期/范围]数据分析已完成，核心结论如下：
🔹 **关键结果**：[指标A] 达 [数值]（环比/同比 [变化]%），主要受 [核心因素] 驱动。
🔹 **异常/洞察**：[指标B] 出现 [上升/下降]，定位到 [具体环节/原因]，需重点关注。
🔹 **目标对齐**：当前进度 [达成/滞后] [X]%，与预期对比 [简要说明]。

💡 **建议动作**：
1. [可执行建议1，如：优化XX流程/追加XX渠道预算]
2. [可执行建议2，如：暂停XX策略/启动AB测试]

📌 **下一步**：[负责人] 将于 [时间] 输出 [具体产出]，详细数据及图表见：[链接/文档名]。欢迎补充视角或同步进展，谢谢！

---
💡 **使用提示**：群消息建议控制在 **5行核心结论+2条行动项** 内，避免信息过载。提供具体数据后，我可秒级为你生成定制版。


In [20]:
# 这一段是在测试：给 basic context 后，模型会怎么回答

answer_with_basic_context = call_llm(
    user_question=current_question,
    system_prompt=basic_context
)

display(Markdown(f"""
### 🤖 有 Basic Context 的回答

{answer_with_basic_context}
"""))


### 🤖 有 Basic Context 的回答

本月销售数据分析结论如下：整体销售额环比上涨12.5%。华东区增长最快（+18.2%），但退货率达7.8%，高于区域平均水平；华南区表现平稳，客单价微降。建议下一步重点跟进华东区退货原因，结合产品批次、渠道反馈及售后记录开展初步排查。请相关同事关注。



## Part 2 小结

到这里，我们已经完成了三件事：

1. 读取了 history  
2. 读取了 memory  
3. 把 current question、history、memory 拼成了 basic context  

现在这个职业数字人已经比 Day3 更进一步了。

Day3 的它会调用工具。  
现在的它开始知道：

**刚才发生了什么，用户是谁，回答时要注意什么。**

下一部分，我们再处理 resource。

resource 不会直接全部塞进 context。

我们会用一个最小 RAG 流程，  
从 resource 里找出和当前问题最相关的资料，  
再放进 context。

# Part 3：给 Resource 加上最小 RAG

上一部分，我们已经把两类 context 放进来了：

```text
当前问题 + history + memory = basic context
```

这一部分，我们处理第三类 context：

resource = 外部资料

但 resource 不能全部直接塞进模型。

所以我们要做一个最小 RAG：

读取资料 → 切成 chunks → 做 embedding → 检索相关内容 → 放进 context

这一部分结束后，我们会得到：

当前问题
+ history
+ memory
+ retrieved resource
= final context

## 1. 读取 Resource 文件

resource 是这次任务需要参考的外部资料。

在这个 Lab 里，我们准备了三类资料：

```text
project_message_template.txt  = 项目群消息格式
metric_explanation.txt        = 指标解释
communication_policy.txt      = 沟通与保密规则

In [21]:

# 这一段是在读取 resource 文件夹里的资料

def load_resource_files(resource_files):
    """
    读取多个 resource 文件，并合并成一个列表。

    返回格式：
    [
        {
            "source": 文件名,
            "content": 文件内容
        }
    ]
    """
    resources = []

    for file_path in resource_files:
        file_path = Path(file_path)

        if not file_path.exists():
            raise FileNotFoundError(f"找不到文件：{file_path}")

        content = file_path.read_text(encoding="utf-8")

        resources.append({
            "source": file_path.name,
            "content": content
        })

    return resources


resources = load_resource_files(resource_files)

display(Markdown(f"""
### ✅ Resource 读取成功

共读取到 {len(resources)} 个 resource 文件：

```text
{chr(10).join(item["source"] for item in resources)}
```

"""))


### ✅ Resource 读取成功

共读取到 3 个 resource 文件：

```text
project_message_template.txt
metric_explanation.txt
communication_policy.txt
```



## 2. 把 Resource 切成 Chunks

接下来，我们把资料切成小块。

这个小块就叫 **chunk**。

你可以把它理解成：

**把一份长资料切成一张张资料卡片。**

后面系统不会直接拿整份资料去问模型，  
而是先从这些小卡片里找最相关的几张。

In [22]:
# 这一段是在把 resource 切成 chunks

def chunk_text(text: str, max_chars: int = 300) -> list:
    """
    把一段长文本切成多个小块。

    参数：
    - text：原始文本
    - max_chars：每个 chunk 最大字符数

    返回：
    - chunks：文本小块列表
    """
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]

    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:
        if len(current_chunk) + len(paragraph) <= max_chars:
            current_chunk += paragraph + "\n"
        else:
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
            current_chunk = paragraph + "\n"

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks


def build_chunks_from_resources(resources, max_chars=300):
    """
    把多个 resource 文件切成 chunks，并保留来源信息。
    """
    all_chunks = []

    for item in resources:
        source = item["source"]
        content = item["content"]

        chunks = chunk_text(content, max_chars=max_chars)

        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "source": source,
                "chunk_id": i + 1,
                "text": chunk
            })

    return all_chunks


chunks = build_chunks_from_resources(resources, max_chars=300)

display(Markdown(f"""
### ✅ Chunk 切分完成

共得到 {len(chunks)} 个 chunk。

下面是前 3 个 chunk 示例：

```text
{chr(10).join([f"[{item['source']} - chunk {item['chunk_id']}]\n{item['text']}\n" for item in chunks[:3]])}
```

"""))


### ✅ Chunk 切分完成

共得到 4 个 chunk。

下面是前 3 个 chunk 示例：

```text
[project_message_template.txt - chunk 1]
项目群消息模板
适用场景：
用于把数据分析结果、项目进展、风险提醒同步到项目群。
推荐格式：
【结论】
用一句话说明当前最重要的发现。
【风险】
说明需要关注的问题。只写可公开同步的概括信息，不展开敏感明细。
【建议动作】
给出下一步建议，最好明确到可以执行的动作。
写作要求：
1. 先说结论，不要铺垫太长。
2. 语气正式、简洁、适合工作群。
3. 不要使用夸张表达。
4. 不要直接公开客户名称、内部成本、利润率和具体异常明细。
5. 如果原因还没有确认，用“建议进一步排查”“需要继续确认”这类表达。

[metric_explanation.txt - chunk 1]
指标解释说明
销售额：
指当前统计周期内完成的销售收入。销售额上涨通常说明整体成交规模扩大，但还需要结合退货率、客单价和渠道结构一起判断。
环比增长：
指当前周期与上一个周期相比的变化。比如本月比上月增长 12.5%，就是环比增长 12.5%。
退货率：
指退货订单或退货金额占总订单或总销售额的比例。退货率偏高可能说明产品质量、交付体验、渠道管理或客户预期存在问题。
客单价：
指平均每笔订单的金额。客单价下降不一定代表业务变差，也可能和促销、产品组合变化或客户结构变化有关。
异常指标：
指明显偏离正常水平的数据。异常指标需要进一步结合业务背景确认，不能只根据数字直接下结论。

[communication_policy.txt - chunk 1]
业务沟通与信息边界说明
项目群可以同步：
1. 总体趋势。
2. 初步结论。
3. 需要关注的风险点。
4. 下一步建议动作。
5. 不涉及敏感细节的概括性提醒。
项目群不建议直接同步：
1. 客户真实名称。
2. 内部成本、利润率、报价底线。
3. 未经确认的异常原因。
4. 具体责任归因。
5. 个人信息或可识别个人的数据。
6. 可能引发误解的单点数据。
推荐表达方式：
1. 用“数据显示”“初步看”“建议进一步确认”表达尚未完全确认的结论。
2. 用“某区域”“部分渠道”“个别指标”替代敏感明细。
3. 如果需要详细数据，应建议在线下报告或专项会议中展开。

```



## 3. 给 Chunks 做 Embedding

现在我们要给每个 chunk 做 embedding。

前面说过：

**embedding = 文字的数字坐标。**

做完 embedding 后，系统就可以比较：

```text
用户问题
和
哪几段资料
```

在意思上更接近。

In [23]:

# 这一段是在给每个 chunk 做 embedding，建立最小 resource index

def build_resource_index(chunks):
    """
    给每个 chunk 生成 embedding。

    返回：
    [
        {
            "source": 文件名,
            "chunk_id": chunk 编号,
            "text": chunk 文本,
            "embedding": chunk 的向量
        }
    ]
    """
    index = []

    for item in chunks:
        embedding = get_embedding(item["text"])

        index.append({
            "source": item["source"],
            "chunk_id": item["chunk_id"],
            "text": item["text"],
            "embedding": embedding
        })

    return index


resource_index = build_resource_index(chunks)

display(Markdown(f"""
### ✅ Resource Index 建立完成

共为 {len(resource_index)} 个 chunk 生成了 embedding。

每个 chunk 现在都有：

```text
source
chunk_id
text
embedding
```

"""))


### ✅ Resource Index 建立完成

共为 4 个 chunk 生成了 embedding。

每个 chunk 现在都有：

```text
source
chunk_id
text
embedding
```




## 4. 计算问题和资料的相似度

现在用户提出一个问题。

系统会把这个问题也变成 embedding。

然后拿“问题的 embedding”  
去和每个 chunk 的 embedding 做比较。

比较结果就是相似度。

相似度越高，说明这段资料越可能和当前问题相关。

In [24]:
# 这一段是在定义相似度计算函数

def cosine_similarity(vec1, vec2):
    """
    计算两个向量的余弦相似度。

    返回值越大，说明两个向量越相似。
    """
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    denominator = np.linalg.norm(vec1) * np.linalg.norm(vec2)

    if denominator == 0:
        return 0

    return np.dot(vec1, vec2) / denominator

## 5. 检索相关 Chunks

接下来，我们写一个检索函数。

它会做三件事：

1. 把用户问题变成 embedding  
2. 和每个 resource chunk 计算相似度  
3. 取最相关的前几段资料  

这里我们先用 `top_k`。

`top_k` 的意思是：

**取相似度最高的前 k 段。**

它不是唯一的筛选方法，  
但很适合用来跑通最小版本。

In [25]:
# 这一段是在根据用户问题检索最相关的 resource chunks

def retrieve_relevant_chunks(question: str, resource_index: list, top_k: int = 3):
    """
    根据用户问题，从 resource index 中检索最相关的 chunks。
    """
    question_embedding = get_embedding(question)

    scored_chunks = []

    for item in resource_index:
        score = cosine_similarity(question_embedding, item["embedding"])

        scored_chunks.append({
            "source": item["source"],
            "chunk_id": item["chunk_id"],
            "text": item["text"],
            "score": score
        })

    scored_chunks = sorted(
        scored_chunks,
        key=lambda x: x["score"],
        reverse=True
    )

    return scored_chunks[:top_k]


retrieved_chunks = retrieve_relevant_chunks(
    question=current_question,
    resource_index=resource_index,
    top_k=3
)

display(Markdown(f"""
### ✅ Retrieved Resource

系统找到的前 3 段相关资料：

```text
{chr(10).join([f"[{item['source']} - chunk {item['chunk_id']}] 相似度：{item['score']:.4f}\n{item['text']}\n" for item in retrieved_chunks])}
```

"""))


### ✅ Retrieved Resource

系统找到的前 3 段相关资料：

```text
[project_message_template.txt - chunk 1] 相似度：0.7639
项目群消息模板
适用场景：
用于把数据分析结果、项目进展、风险提醒同步到项目群。
推荐格式：
【结论】
用一句话说明当前最重要的发现。
【风险】
说明需要关注的问题。只写可公开同步的概括信息，不展开敏感明细。
【建议动作】
给出下一步建议，最好明确到可以执行的动作。
写作要求：
1. 先说结论，不要铺垫太长。
2. 语气正式、简洁、适合工作群。
3. 不要使用夸张表达。
4. 不要直接公开客户名称、内部成本、利润率和具体异常明细。
5. 如果原因还没有确认，用“建议进一步排查”“需要继续确认”这类表达。

[communication_policy.txt - chunk 1] 相似度：0.6570
业务沟通与信息边界说明
项目群可以同步：
1. 总体趋势。
2. 初步结论。
3. 需要关注的风险点。
4. 下一步建议动作。
5. 不涉及敏感细节的概括性提醒。
项目群不建议直接同步：
1. 客户真实名称。
2. 内部成本、利润率、报价底线。
3. 未经确认的异常原因。
4. 具体责任归因。
5. 个人信息或可识别个人的数据。
6. 可能引发误解的单点数据。
推荐表达方式：
1. 用“数据显示”“初步看”“建议进一步确认”表达尚未完全确认的结论。
2. 用“某区域”“部分渠道”“个别指标”替代敏感明细。
3. 如果需要详细数据，应建议在线下报告或专项会议中展开。

[communication_policy.txt - chunk 2] 相似度：0.6321
4. 发群消息时，优先保证清楚、稳妥、可执行。

```




## 6. 把 Retrieved Resource 加进 Context

现在，我们已经有了三类信息：

```text
history
memory
retrieved resource
```

接下来，把它们拼成 final context。

这一版 context 就比上一部分更完整了：

```text
当前问题
+ history
+ memory
+ retrieved resource
= final context
```

这就是 Day4 的最小闭环。

接下来，把它们拼成 final context。

这一版 context 就比上一部分更完整了：


当前问题 + history + memory + retrieved resource = final context

这就是 Day4 的最小闭环。

In [26]:

# 这一段是在把 history、memory、retrieved resource 拼成 final context

def format_retrieved_chunks(retrieved_chunks: list) -> str:
    """
    把检索到的 chunks 整理成适合放进 prompt 的文字。
    """
    lines = []

    for item in retrieved_chunks:
        lines.append(
            f"来源：{item['source']}，chunk {item['chunk_id']}\n{item['text']}"
        )

    return "\n\n".join(lines)


def build_final_context(
    current_question: str,
    history_text: str,
    memory: dict,
    retrieved_chunks: list
) -> str:
    """
    构建最终 context。

    包含：
    - 当前问题
    - history
    - memory
    - retrieved resource
    """
    memory_text = format_memory(memory)
    retrieved_resource_text = format_retrieved_chunks(retrieved_chunks)

    final_context = f"""
    你是一个职业数字人助手。

    请你根据下面的 context 回答用户问题。

    【当前问题】
    {current_question}

    【History：当前对话前面发生过什么】
    {history_text}

    【Memory：长期需要记住的信息】
    {memory_text}

    【Retrieved Resource：这次任务检索到的相关资料】
    {retrieved_resource_text}

    请注意：
    1. 回答必须基于上面的 context。
    2. 如果要生成群消息，请遵守 retrieved resource 里的格式和沟通规则。
    3. 请结合 memory 中的用户角色、表达风格和信息边界。
    4. 不要展开客户名称、内部成本、利润或异常明细。
    5. 不要编造 context 里没有的信息。
    """

    return final_context.strip()


final_context = build_final_context(
    current_question=current_question,
    history_text=history_text,
    memory=memory,
    retrieved_chunks=retrieved_chunks
)

display(Markdown(f"""
### ✅ Final Context 已生成

```text
{final_context}
```
"""))



### ✅ Final Context 已生成

```text
你是一个职业数字人助手。

    请你根据下面的 context 回答用户问题。

    【当前问题】
    请把刚才的数据分析结果整理成一段适合发到项目群的消息。

    【History：当前对话前面发生过什么】
    Day3 数据分析对话历史

用户上传了一份销售数据表，文件名为 sales_data.csv。

数据分析工具完成了基础分析，得到以下结果：
1. 本月整体销售额较上月上涨 12.5%。
2. 华东区销售额增长最快，环比增长 18.2%。
3. 华东区退货率也明显偏高，达到 7.8%，高于其他区域平均水平。
4. 华南区销售额稳定，但客单价略有下降。
5. 数据分析工具建议继续关注华东区退货原因，尤其是产品批次、渠道反馈和售后记录。

用户接着说：
“把刚才的数据分析结果，整理一下发到项目群里。”

注意：
这份 history 只是为了 Lab4 演示。
它代表当前对话前面已经发生过的内容。


    【Memory：长期需要记住的信息】
    - user_profile: {'role': '项目负责人', 'department': '数字化与业务协同团队', 'work_context': '经常需要把数据分析结果整理成适合项目群、领导群或跨部门会议使用的简短结论。', 'communication_style': '简洁、正式、清楚，不夸张，不写空话。'}
- long_term_preferences: {'message_style': '先说结论，再说风险，最后说建议动作。', 'preferred_length': '适合发群消息，控制在 150 字以内。', 'tone': '稳重、专业、便于团队快速理解。'}
- boundaries: {'sensitive_information': ['客户名称', '内部成本', '利润率', '具体异常明细', '未经确认的责任归因', '个人信息'], 'rules': ['敏感数据不要直接发到群里。', '涉及客户、成本、利润和异常明细时，只做概括表达。', '没有确认原因前，不要直接判断责任。', '如果结论来自初步分析，要保留谨慎表达。']}

    【Retrieved Resource：这次任务检索到的相关资料】
    来源：project_message_template.txt，chunk 1
项目群消息模板
适用场景：
用于把数据分析结果、项目进展、风险提醒同步到项目群。
推荐格式：
【结论】
用一句话说明当前最重要的发现。
【风险】
说明需要关注的问题。只写可公开同步的概括信息，不展开敏感明细。
【建议动作】
给出下一步建议，最好明确到可以执行的动作。
写作要求：
1. 先说结论，不要铺垫太长。
2. 语气正式、简洁、适合工作群。
3. 不要使用夸张表达。
4. 不要直接公开客户名称、内部成本、利润率和具体异常明细。
5. 如果原因还没有确认，用“建议进一步排查”“需要继续确认”这类表达。

来源：communication_policy.txt，chunk 1
业务沟通与信息边界说明
项目群可以同步：
1. 总体趋势。
2. 初步结论。
3. 需要关注的风险点。
4. 下一步建议动作。
5. 不涉及敏感细节的概括性提醒。
项目群不建议直接同步：
1. 客户真实名称。
2. 内部成本、利润率、报价底线。
3. 未经确认的异常原因。
4. 具体责任归因。
5. 个人信息或可识别个人的数据。
6. 可能引发误解的单点数据。
推荐表达方式：
1. 用“数据显示”“初步看”“建议进一步确认”表达尚未完全确认的结论。
2. 用“某区域”“部分渠道”“个别指标”替代敏感明细。
3. 如果需要详细数据，应建议在线下报告或专项会议中展开。

来源：communication_policy.txt，chunk 2
4. 发群消息时，优先保证清楚、稳妥、可执行。

    请注意：
    1. 回答必须基于上面的 context。
    2. 如果要生成群消息，请遵守 retrieved resource 里的格式和沟通规则。
    3. 请结合 memory 中的用户角色、表达风格和信息边界。
    4. 不要展开客户名称、内部成本、利润或异常明细。
    5. 不要编造 context 里没有的信息。
```


## 7. 用 Final Context 生成回答

现在，我们把 final context 交给模型。

这一次，模型看到的不只是用户问题。

它还看到了：

```text
history：刚才发生了什么
memory：用户是谁，要注意什么
retrieved resource：这次任务要参考哪些资料
```

所以它的回答会更贴近真实工作场景。

In [27]:
# 这一段是在用 final context 调用模型，生成回答

final_answer = call_llm(
    user_question=current_question,
    system_prompt=final_context
)

display(Markdown(f"""
### 🤖 Final Answer

{final_answer}
"""))


### 🤖 Final Answer

【结论】本月整体销售额环比上涨12.5%。华东区增速领先（+18.2%），华南区销售平稳但客单价微降。
【风险】华东区退货率达7.8%，高于区域平均水平，需关注对业务指标的潜在影响。
【建议动作】建议协同业务与售后团队进一步排查华东区退货原因，华南区同步跟踪客单价趋势。详细明细建议线下专项复盘确认。


# Part 4：总结与进阶练习

到这里，Day 4 的最小闭环已经完成了。

我们已经让职业数字人看到三类 context：

```text
history  = 刚刚发生了什么
memory   = 用户是谁、有什么长期偏好和边界
resource = 这次任务要参考哪些外部资料
```

最后，我们把它们组合成：

当前问题 + history + memory + retrieved resource = final context

这就是 Day 4 最重要的成果。

职业数字人不只是会调用工具了。
它开始能接住前面对话，参考长期规则，也能基于外部资料回答。

进阶练习

如果你想继续往前做，可以尝试下面 3 个小任务。

## 练习 1：自动更新 history

现在我们的 history 是提前写好的。

你可以尝试让系统在每轮对话结束后，自动把内容追加到 history 文件里：

用户问题
AI 回答
时间

这样下一轮对话时，AI 就能接住前面发生过什么。

## 练习 2：让 memory 需要确认后再写入

memory 不适合随便自动更新。

你可以尝试做一个小判断：

这条信息是否值得长期记住？

如果值得，再让用户确认：

是否要把这条信息写入 memory？

确认后，再更新 memory 文件。

## 练习 3：给 resource 增加更多资料

你可以继续往 data/context/resource/ 里放更多文件，比如：

项目说明
会议纪要
周报模板
指标解释
沟通规范
FAQ

然后重新运行 RAG 流程，观察检索出来的资料会不会变化。

拓展阅读关键词

## 拓展阅读：

如果你想继续查资料，可以搜索这些关键词：

RAG
embedding
vector database
conversation history
AI memory
context window

今天先到这里。

Day 4 的重点不是把 RAG 做复杂，
而是先理解：

AI 的回答质量，很大程度取决于它这一次看到了什么。